## Design minimal peptide library example notebook: 
Minimal peptides are defined as all 8-11-mer peptides containing at least one mutated residue, for SNVs, in-frame insertions, and frameshifting indels. For in-frame deletions, minimal peptides are required to contain both amino acids forming the neojunction created by the deletion. Binding confidence (%Rank) to all patient HLA class I alleles is predicted using [MixMHCpred3](https://github.com/GfellerLab/MixMHCpred). Peptides with %Rank_bestAllele >2% are filtered to reduce the minimal peptide sorting set size. Protein TPM mRNA expression and %Rank_bestAllele are converted to percentile ranks, where 1.0 represents the highest TPM and lowest %Rank_bestAllele. A combined sort score is calculated as 0.2 × TPM percentile rank + 0.8 × %Rank_bestAllele percentile rank, and minimal peptides are sorted by this score. Duplicate minimal peptide amino acid sequences are then removed, retaining the first-ranked sequence. The top minimal peptides, as based on weighted predicted MHC class I binding and RNA expression, are selected.

Selected minimal peptide-encoding library oligonucleotide pools are prepared in this example notebook. Minimal peptide amino acid sequences are codon optimized with the TCRtoolbox codon optimizer `codon_optimize`. Briefly, codon optimization removes type IIS BsmBI recognition sites, four-base Golden Gate overhangs, Kozak sequences, primer binding sites, repeats longer than four nucleotides, and terminator motifs (TTTTT or AAAAA). In addition, codon optimization enforces a global GC content between 35–60%, as [previously described](https://www.biorxiv.org/content/10.1101/2025.04.28.651095v3.article-info). `write_barcoded_p12_p20_epitope_oligo_order_fasta` is then used to add a TAA stop-codon directly after each codon-optimized coding sequence, followed by a unique 36 bp (minimal peptide libraries) DNA barcode. The resulting nucleotide sequences are flanked by BsmBI recognition sites and orthogonal primer binding sites (stored as constants in the TCRtoolbox software) for PCR amplification. Multiple patient libraries can be encoded as subpools in a single oligonucleotide pool by flanking these with unique orthogonal primer combinations, as further described in the docstring of `write_barcoded_p12_p20_epitope_oligo_order_fasta`. To identify design errors before wet lab cloning, cloning of each minimal peptide is modeled in silico using `check_p12_p20_oligo_order_fasta_pydna`, which uses [pydna](https://github.com/pydna-group/pydna), modeling restriction digestion, DNA ligation, and PCR.


In [ ]:
import collections
from pathlib import Path

import numpy as np
import pandas as pd
import pysam
from Bio.Seq import Seq

from tcr_toolbox.epi_assembly.epi_parse_utils import align_to_add_var_start_and_end_aa_seq_ag_df
from tcr_toolbox.epi_assembly.mixmhcpred import (
    parse_mixmhcpred_out_txt_list,
    run_mixmhcpred_patient_df_dict,
)
from tcr_toolbox.epi_assembly.p12_p20_vector_utils import (
    check_p12_p20_oligo_order_fasta_pydna,
    write_barcoded_p12_p20_epitope_oligo_order_fasta,
    write_minigene_aa_fasta_from_oligo_order_fasta,
)
from tcr_toolbox.tcr_assembly.constants import plate_primers_dict, sites_to_check
from tcr_toolbox.utils.codon_optimizer import codon_optimize

First install MixMHCpred executable: https://github.com/GfellerLab/MixMHCpred

In [ ]:
mixmhcpred_exe = "/MixMHCpred/MixMHCpred"

In [ ]:
patient_list = ["patient01", "patient02"]
run_dir = Path("/minigene_assembly")

neo_ag_run_dir = run_dir / "datasets" / "neoantigen"
patient_hla_df = pd.read_csv(run_dir / "patient_hla_alleles.csv")
patient_file_list = [neo_ag_run_dir / f"{patient}_neoantigen_candidates.xlsx" for patient in patient_list]

In [5]:
patient_hla_df

,patient,gene,allele_1_mix_enc,allele_2_mix_enc
0,patient01,HLA-A,A0201,A2402
1,patient01,HLA-B,B0702,B4901
2,patient01,HLA-C,C0727,C0763
3,patient02,HLA-A,A0201,A2601
4,patient02,HLA-B,B1553,B4037
5,patient02,HLA-C,C0304,C0304


In [ ]:
ag_df_dict = {}
for patient_xlsx in patient_file_list:
    patient = patient_xlsx.stem.split("_")[0]
    print(patient)
    ag_df_dict[patient] = pd.read_excel(patient_xlsx)

    # mutation start and end position annotation is needed to only generate kmer peptides that correctly overlap the mutation
    ag_df_dict[patient] = align_to_add_var_start_and_end_aa_seq_ag_df(
        ag_df=ag_df_dict[patient],
        aa_var_seq_col="minigene_mutated_aa_seq",
        aa_ref_seq_col="minigene_wt_aa_seq"
    )

patient01
260
258
260
259
patient02
68
65
68
68


In [ ]:
mixmhcpred_out_txt_list = run_mixmhcpred_patient_df_dict(
    patient_df_dict=ag_df_dict,
    patient_hla_df=patient_hla_df,
    run_dir=neo_ag_run_dir,
    kmer_list=[8, 9, 10, 11],
    gene_id_col="ag_name",
    aa_seq_col="minigene_mutated_aa_seq",
    hla_df_patient_col="patient",
    mixmhcpred_exe=mixmhcpred_exe,
    allele_1_col="allele_1_mix_enc",
    allele_2_col="allele_2_mix_enc",
    variant=True,
    minigene_aa_var_start_col="minigene_aa_var_start",
    minigene_aa_var_end_col="minigene_aa_var_end",
    var_type_for_kmer_overlap_col="var_type_for_kmer_overlap_col",
)

Processing: patient01
A0201,B0702,C0727,A2402,B4901,C0763
Processing: patient02
A0201,B1553,C0304,A2601,B4037,C0304


In [ ]:
parsed_neo_csv_list = parse_mixmhcpred_out_txt_list(
    mixmhcpred_out_txt_list=mixmhcpred_out_txt_list,
    patient_df_dict=ag_df_dict,
    gene_id_col="ag_name",
    mrna_expression_col="TPM",
    keep_top_number_peptides=350,
    rank_by_binning=False,
    include_cols_patient_df_list=None,
    mrna_sorting_weight=0.2,
    percent_rank_threshold=2.0,
)

In [ ]:
parsed_neo_dict = dict()
for parsed_csv in parsed_neo_csv_list:
    patient = parsed_csv.stem.split("_")[0]
    print(patient)
    parsed_neo_dict[patient] = pd.read_csv(parsed_csv, index_col=0)
    parsed_neo_dict[patient]["kmer_name_hla"] = parsed_neo_dict[patient]["kmer_name"] + "_" + parsed_neo_dict[patient]["BestAllele"]
    print(f"Number of minimal peptides parsed: {parsed_neo_dict[patient].shape[0]}")
    parsed_neo_dict[patient].to_csv(neo_ag_run_dir / f"outs/neo_{patient}_mixmhcpred_parsed.csv", index=False)

In [ ]:
model_viral_df = pd.read_excel(
    "/viral_and_model_epitopes.xlsx",
    usecols=np.arange(0, 6),
)

In [29]:
model_viral_df

,ag_name,minigene_aa,HLA_allele,epitope_name,epitope_aa,epitope_name_hla
0,s_MART1_EAA,HGHSYTTAEEAAGIGILTVILGVLLLIGCWY,A0201,MART1_EAA,EAAGIGILTV,MART1_EAA_A0201
1,s_MART1_ELA,HGHSYTTAEELAGIGILTVILGVLLLIGCWY,A0201,MART1_ELA,ELAGIGILTV,MART1_ELA_A0201
2,s_NYESO1_TQA,LQLSISSCLQQLSLLMWITQAFLPVFLAQPP,A0201,NYESO1,SLLMWITQA,NYESO1_A0201
3,MEL063_CCSER2_P329L,GYRMVHPSLLKSSRSLFSGTMTVDGNKNSPA,A0201,MEL063_CCSER2_P329L,GYRMVHPSLLKSSRSLFSGTMTVDGNKNSPA,MEL063_CCSER2_P329L_A0201
4,s_CMV_pp65_NLV,VFTWPPWQAGILARNLVPMVATVQGQNLKYQ,A0201,NLV,NLVPMVATV,NLV_A0201
5,s_FLU_MP1_GIL,RPILSPLTKGILGFVFTLTVPSERGLQRRRF,A0201,GIL,GILGFVFTL,GIL_A0201
6,s_EBV_BMLF1_GLC,LDARMQAIQNAGLCTLVAMLEETIFWLQEIT,A0201,GLC,GLCTLVAML,GLC_A0201
7,s_EBV_BZLF1_LPC,YQDLGGPSQAPLPCVLWPVLPEPLPQGQLTA,B0702,LPC,LPCVLWPVL,LPC_B0702
8,s_CMV_pp65_TPR,TSGSDSDEELVTTERKTPRVTGGGAMASAST,B0702,TPR,TPRVTGGGAM,TPR_B0702
9,s_CMV_pp65_VIH,ADAVIHASGKQMWQARLTVSGLAWTRQQNQW,A2601,VIH,VIHASGKQM,VIH_A2601


In [ ]:
# This is an example of how to add additional epitopes that can be used as a positive control or to e.g., investigate viral reactivity to a minimal peptide library: 
model_viral_select_mini_screen_dict = {
    "patient01": [
        "MEL063_CCSER2_P329L", # weakly potent positive-control neoAg, minimal peptide restricted to HLA-A*02:01. Detecting it suggests your screen was sensitive enough to also catch other neoAgs.
        "s_MART1_EAA",
        "s_MART1_ELA",
        "s_NYESO1_TQA",
        "s_CMV_pp65_NLV",
        "s_FLU_MP1_GIL",
        "s_EBV_BMLF1_GLC",
        "s_FLU_PB1_RYG",
        "s_EBV_BRLF1_TYP",
        "s_CMV_pp65_QYD",
    ],
    "patient02": ["MEL063_CCSER2_P329L", "s_MART1_EAA", "s_MART1_ELA", "s_NYESO1_TQA", "s_CMV_pp65_NLV", "s_FLU_MP1_GIL", "s_EBV_BMLF1_GLC", "s_CMV_pp65_VIH", "s_FLU_PB1_YSH"],
}
model_viral_select_pep_screen_dict = {
    "patient01": ["MEL063_CCSER2_P329L", "NLV", "GIL", "GLC", "RYG", "TYP", "QYD"],
    "patient02": ["MEL063_CCSER2_P329L", "NLV", "GIL", "GLC", "VIH", "YSH"],
}

In [ ]:
patient_pep_name_list_dict = collections.defaultdict(list)
patient_pep_seq_list_dict = collections.defaultdict(list)

selected_patient_list = ["patient01", "patient02"]
for patient in selected_patient_list:
    print(patient)

    patient_pep_name_list_dict[patient] = (
        parsed_neo_dict[patient]["kmer_name_hla"].str.replace("-", "_").tolist()
        + model_viral_df.loc[model_viral_df["epitope_name"].isin(model_viral_select_pep_screen_dict[patient]), "epitope_name_hla"].tolist()
    )
    patient_pep_seq_list_dict[patient] = (
        parsed_neo_dict[patient]["Peptide"].tolist()
        + model_viral_df.loc[model_viral_df["epitope_name"].isin(model_viral_select_pep_screen_dict[patient]), "epitope_aa"].tolist()
    )

    if not len(patient_pep_name_list_dict[patient]) == (parsed_neo_dict[patient].shape[0] + len(model_viral_select_pep_screen_dict[patient])):
        raise Exception("Name list out of sync with shape of parsed_neo_dict!")
    if not len(patient_pep_seq_list_dict[patient]) == (parsed_neo_dict[patient].shape[0] + len(model_viral_select_pep_screen_dict[patient])):
        raise Exception("Sequence list out of sync with shape of parsed_neo_dict!")

    print(f"Number of unique minimal peptides ordered: {len(patient_pep_seq_list_dict[patient])}")
    nan_indices = [i for i, x in enumerate(patient_pep_seq_list_dict[patient]) if pd.isna(x)]
    if nan_indices:
        print(f"NaN indices found in aa sequence order list: {nan_indices}")

    print("\n")

patient01
Number of unique minimal peptides ordered: 382


patient02
Number of unique minimal peptides ordered: 319




In [ ]:
pep_nt_seq_optimized_dict = collections.defaultdict(lambda: collections.defaultdict(str))
pep_nt_seq_cannot_be_optimized_dict = collections.defaultdict(list)
for patient in selected_patient_list:
    print(patient)
    for name, aa_seq in zip(patient_pep_name_list_dict[patient], patient_pep_seq_list_dict[patient]):
        pep_nt_seq_optimized = codon_optimize(
            sequence_aa=aa_seq, enzymes=["BsmBI"], sites_to_check=sites_to_check, if_fail="return_none", verbose=1, steepness=4
        )

        if isinstance(pep_nt_seq_optimized, str):
            pep_nt_seq_optimized_dict[patient][name + "_1"] = pep_nt_seq_optimized  # rep_1
            pep_nt_seq_optimized_dict[patient][name + "_2"] = pep_nt_seq_optimized  # rep_2
        else:
            pep_nt_seq_cannot_be_optimized_dict[patient].append(name)
    print(pep_nt_seq_cannot_be_optimized_dict[patient])
    print(len(pep_nt_seq_optimized_dict[patient]))
    print("\n\n")

patient01
[]
764



patient02
[]
638





In [ ]:
# To save costs, patients screened simultaneously can each be added as a subpool to the
# minimal peptide-encoding oligonucleotide library. Each patient's subpool is flanked by an
# orthogonal primer pair, so it can later be amplified out of the pool individually to generate
# the dsDNA that is needed anyway for Golden Gate assembly into the P12 or P20 lentiviral vector.

# Skip this cell if you do not have subpools! 
pep_subpool_indices_dict = collections.defaultdict(list)
end_idx = 0
for subpool, patient in enumerate(pep_nt_seq_optimized_dict.keys()):
    if subpool == 0:
        start_idx = 0
        end_idx = len(pep_nt_seq_optimized_dict[patient])
        pep_subpool_indices_dict[subpool] = np.arange(start_idx, end_idx)
    else:
        start_idx = end_idx
        end_idx += len(pep_nt_seq_optimized_dict[patient])
        pep_subpool_indices_dict[subpool] = np.arange(start_idx, end_idx)

pep_nt_seq_list = [pep_nt_seq for patient_dict in pep_nt_seq_optimized_dict.values() for pep_nt_seq in patient_dict.values()]
pep_name_list = [pep_name for patient in pep_nt_seq_optimized_dict.keys() for pep_name in pep_nt_seq_optimized_dict[patient].keys()]

# Check whether subpool indices assignment worked using sets:
for subpool, patient in enumerate(pep_nt_seq_optimized_dict.keys()):
    print(patient)
    print(
        len(set(pep_nt_seq_optimized_dict[patient].keys()).intersection(set(np.array(pep_name_list)[pep_subpool_indices_dict[subpool]])))
        == len(set(pep_nt_seq_optimized_dict[patient].keys()))
    )
    print(
        len(set(pep_nt_seq_optimized_dict[patient].values()).intersection(set(np.array(pep_nt_seq_list)[pep_subpool_indices_dict[subpool]])))
        == len(set(pep_nt_seq_optimized_dict[patient].values()))
    )

patient01
True
True
patient02
True
True


In [ ]:
minimal_peptide_run_dir = run_dir / "assembly_runs" / "minimal_peptide"
order_fasta_out_file = minimal_peptide_run_dir / "minimal_peptide_barcoded_p20_oligo_order.fasta"

In [ ]:
write_barcoded_p12_p20_epitope_oligo_order_fasta(
    epitopes_nucl_list=pep_nt_seq_list, # Now list is made in cell above because 2 subpools. Use list(pep_nt_seq_optimized_dict.values()) if no subpools are used. 
    epitopes_names_list=[kmer_name.replace ("-", "_") for kmer_name in pep_name_list], # Now list is made in cell above because 2 subpools. Use [kmer_name.replace("-", "_") for kmer_name in pep_nt_seq_optimized_dict.keys()] if no subpools are used.
    primers_fw_list=[plate_primers_dict[i][0] for i in np.arange(0, 2)], # Now set to 2 subpools. Set to [plate_primers_dict[i][0] for i in np.arange(0, 1)] if no subpools are used. 
    primers_rv_list=[plate_primers_dict[i][1] for i in np.arange(0, 2)], # Now set to 2 subpools. Set to [plate_primers_dict[i][1] for i in np.arange(0, 1)] if no subpools are used. 
    subpool_indices_dict=pep_subpool_indices_dict, # Now subpool indices for 2 subpools. Set to None if no subpools are used. 
    barcode_large_bool=True, # 36 bp barcodes to compensate for the shorter length of minimal peptides
    fasta_out_fname=order_fasta_out_file,
    no_uvp_but_only_subpool_primers=True,
)

Generating barcodes!
Length barcode list after site removal: 168479
Adding subpool primers...
Start processing epitopes for fasta writing!
1000 epitopes processed!
All 1402 epitopes processed!
Fasta file written!


In [ ]:
order_pep_aa_fasta_out_file = minimal_peptide_run_dir / (order_fasta_out_file.stem + "_minimal_peptide_aa_seqs.fasta")
write_minigene_aa_fasta_from_oligo_order_fasta(oligo_order_fasta_out_file=order_fasta_out_file, fasta_out_fname=order_pep_aa_fasta_out_file)

In [ ]:
pep_aa_dict = collections.defaultdict(str)
with pysam.FastxFile(order_pep_aa_fasta_out_file) as fasta_in:
    for entry in fasta_in:
        pep_aa_dict[entry.name] = entry.sequence
print([Seq(pep_nt_seq).translate(to_stop=True) for pep_nt_seq in pep_nt_seq_list] == [Seq(aa_seq) for aa_seq in pep_aa_dict.values()])

True


In [ ]:
# To identify design errors before wet lab cloning, model cloning (PCR, digestion, and ligation) of minimal peptides using pydna: 
check_p12_p20_oligo_order_fasta_pydna(
    fasta_in_fname=order_fasta_out_file,
    primers_fw_list=[plate_primers_dict[i][0] for i in np.arange(0, 2)],
    primers_rv_list=[plate_primers_dict[i][1] for i in np.arange(0, 2)],
    aa_check_list=[Seq(pep_nt_seq).translate(to_stop=True) for pep_nt_seq in pep_nt_seq_list],
    fusion_protein_bool_list=None,
    also_check_full_pool=False,
    epitope_variant_hamming_pool=False,
    no_uvp_but_only_subpool_primers=True
)

Started checking subpools: dict_keys(['0', '1'])
1 oligos correct!
1001 oligos correct!
All 1402 oligos are correct!
Finished checking all oligos!


## FASTA header formats

**MixMHCpred kmer FASTA** (`<run_dir>/mixmhcpred/*.fasta`): each record's header line is `<name>_<k>-<i>`
— `name` = antigen name (`ag_name`), `k` = kmer length, `i` = 0-based start position in the antigen's
amino acid sequence. E.g. `patient01_TP53_R175H_9-12`.

**Final oligo order FASTA** (`order_fasta_out_file`, written by `write_barcoded_p12_p20_epitope_oligo_order_fasta`):
each header line is `<barcode>_<epitope_name>_<subpool>`. `<epitope_name>` is supplied by the caller and
may itself contain additional `_` characters — here it's the kmer name above (`<name>_<k>-<i>`).
